# Importing Libraries


In [2]:

from transformers import  AutoTokenizer, BertForSequenceClassification , DataCollatorWithPadding
import torch 
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)




# Setting up the GPU as device 

In [3]:
if torch.cuda.is_available(): 
    device_type = "cuda"
elif torch.backends.mps.is_available():
    device_type = "mps"
else : device_type = "cpu"
device = torch.device(device_type)
print(device)

mps


# Data importing 


In [4]:

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/stanfordnlp/sst2/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/stanfordnlp/sst2/" + splits["validation"])
df_test =pd.read_parquet("hf://datasets/stanfordnlp/sst2/" + splits["test"])

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")


In [5]:
def tokenize_text(text,tokenizer= tokenizer):
    token = tokenizer(text,
    padding=True,
    truncation=True,
    max_length=512)
    return token

def tokenize_data(df, tokenizer):
    tokens_data = df.copy()
    tokens_data["sentence"] =tokens_data["sentence"].apply(tokenize_text,tokenizer=tokenizer)

    tokens_data["input_ids"] = tokens_data["sentence"].apply(lambda x: x["input_ids"])
    tokens_data["attention_mask"] = tokens_data["sentence"].apply(lambda x: x["attention_mask"])
    tokens_data["token_type_ids"] = tokens_data["sentence"].apply(lambda x: x["token_type_ids"])
    tokens_data=tokens_data.drop(columns= ["sentence"])
    return  tokens_data

train_tokens= tokenize_data(df,tokenizer)
test_tokens= tokenize_data(df_test, tokenizer)
val_tokens= tokenize_data(df_val, tokenizer)
display(train_tokens.head(5))


,idx,label,input_ids,attention_mask,token_type_ids
0,0,0,"[101, 5342, 2047, 3595, 8496, 2013, 1996, 1864...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,1,0,"[101, 3397, 2053, 15966, 1010, 2069, 4450, 209...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,2,1,"[101, 2008, 7459, 2049, 3494, 1998, 10639, 201...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,3,0,"[101, 3464, 12580, 8510, 2000, 3961, 1996, 216...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,4,0,"[101, 2006, 1996, 5409, 7195, 1011, 1997, 1011...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Creating dataset and dataloader 


In [6]:


class SST2Dataset(Dataset):

    def __init__(self, df):
        self.input_ids = df["input_ids"].tolist()
        self.attention_mask = df["attention_mask"].tolist()
        self.token_type_ids = df["token_type_ids"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "token_type_ids": self.token_type_ids[idx],
            "labels": self.labels[idx]
        }


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)
train_dataset = SST2Dataset(df= train_tokens)
test_dataset = SST2Dataset(df= test_tokens)
val_dataset = SST2Dataset(df= val_tokens)


train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

batch= next(iter(train_dataset))
print(batch.items())

dict_items([('input_ids', [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102]), ('attention_mask', [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), ('token_type_ids', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), ('labels', 0)])


In [7]:

model =BertForSequenceClassification.from_pretrained("google-bert/bert-base-uncased",
                                                     num_labels= 2)
model.to(device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14607.65it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

# Optimizer


In [8]:
optimizer =AdamW(
    model.parameters(),
    lr= 2e-5
)


# Training Loop 


In [ ]:

num_epoch = 7

val_loss_curve = []
train_loss_curve = []

for epoch in range(num_epoch):

    print("#####################################################################")
    print(f"Training Epoch {epoch+1}")

    # ================= TRAINING =================
    model.train()
    train_loss = 0

    for batch in tqdm(train_dataloader):

        # Move batch to device
        batch = {key: value.to(device) for key, value in batch.items()}

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"],
            labels=batch["labels"]
        )

        # Calculate loss
        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_dataloader)
    train_loss_curve.append(avg_train_loss)

    print(f"Epoch {epoch + 1}, Training Loss: {avg_train_loss:.4f}")

    # ================= VALIDATION =================
    print("------------------------------------------------------------------")
    print("Validation")

    model.eval()
    val_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in tqdm(val_dataloader):

            batch = {key: value.to(device) for key, value in batch.items()}

            # Forward propagation only
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch["token_type_ids"],
                labels = batch["labels"]
            )
            # Convert logits into predicted classes
            predictions = torch.argmax(outputs.logits, dim=1)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())
            #get Loss
            loss = outputs.loss
            val_loss += loss.item()


    avg_val_loss = val_loss / len(val_dataloader)
    val_loss_curve.append(avg_val_loss)

    print(f"Epoch {epoch + 1}, Validation Loss: {avg_val_loss:.4f}")
    
    
    
    accuracy = accuracy_score(all_labels, all_predictions)

    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

    print(classification_report(
        all_labels,
        all_predictions,
        target_names=["Negative", "Positive"]
    ))

    print(confusion_matrix(all_labels, all_predictions))


#####################################################################
Training Epoch 1


  1%|▏         | 58/4210 [00:11<13:16,  5.22it/s]

In [ ]:
torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
            
            }, "trained_model")